# AIRPRED — Stage 1: Data Preparation (Colab)

This notebook implements the **Stage 1 — Data Preparation** box from your pipeline diagram:

```
NASA MERRA-2 ─┐
              ├─► Merge on Stations + Datetime ─► Cleaning → Normalize ─► Sliding window (L=48) → 70/10/20 split
ERA5 ─────────┘

OpenAQ Manila ─► NASA MERRA-2 bias validation   (separate branch — NOT merged into the training data)
```

**Before running this for real:** two open items from our validation checks so far —
1. Your current MERRA-2 CSVs only contain `SO4CMASS`. If you haven't re-pulled with `SO4SMASS` yet, this notebook will still run, but Cell 5 below has a loud sanity check that will warn you if the derived PM2.5 is physically implausible (it will be, with `SO4CMASS`).
2. Make sure the ERA5 file you upload here is the **re-pulled** one (10 NCR cities: …Mandaluyong, Navotas, Pasay, San_Juan), not the old archipelago-wide file. Cell 8 checks this automatically and will raise an error if the station list is wrong.


## 0. Mount Drive (recommended) or plan to re-upload files each session

Colab's local disk is wiped when the runtime disconnects. Mounting Drive means you
only upload your CSVs once and can rerun this notebook across sessions without
re-uploading. If you'd rather not use Drive, skip this cell and use the
`files.upload()` cells later instead — you'll just need to re-upload each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Suggested layout in your Drive — adjust PROJECT_DIR if you use a different path
PROJECT_DIR = '/content/drive/MyDrive/AIRPRED'
import os
os.makedirs(f'{PROJECT_DIR}/data/raw/merra2', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/raw/era5', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/validation', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/final', exist_ok=True)
print(f"Working directory: {PROJECT_DIR}")

## 1. Get the AIRPRED code onto this runtime

This reuses the exact same `src/preprocessing/preprocessing.py` functions from the
code skeleton — one source of truth, no logic duplicated between your local VS Code
setup and Colab. Upload `AIRPRED_code_skeleton.zip` when prompted.

In [ ]:
from google.colab import files
import zipfile, sys, os

uploaded = files.upload()  # select AIRPRED_code_skeleton.zip
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

sys.path.insert(0, '/content/AIRPRED')
os.chdir('/content/AIRPRED')

from src.preprocessing.preprocessing import (
    derive_pm25_thesis_formula, derive_pm25_nasa_gmao_formula,
    merge_datasets, validate_merged, interpolate_gaps,
    chronological_cutoffs, chronological_split, fit_transform_split,
    build_dataset, compute_bias_metrics, compare_blh_sources, fill_blh_gap,
    FEATURE_COLS, MET_COLS,
)
import pandas as pd
import numpy as np
print("AIRPRED source imported successfully.")

## 2. Load the raw datasets

Upload your 10 MERRA-2 CSVs (`MERRA2_<City>.csv`), the re-pulled ERA5 CSV, and the
OpenAQ CSV. If you mounted Drive in Step 0 and already saved files there, point the
paths at `PROJECT_DIR` instead of using `files.upload()`.

In [ ]:
# --- MERRA-2: upload all 10 city CSVs at once (multi-select in the file picker) ---
from google.colab import files
merra2_uploaded = files.upload()

EXPECTED_CITIES = {"Manila", "Quezon_City", "Caloocan", "Valenzuela", "Pasig",
                    "Makati", "Mandaluyong", "Navotas", "Pasay", "San_Juan"}

merra2_frames = []
for fname in merra2_uploaded:
    df = pd.read_csv(fname, parse_dates=["datetime"])
    # Your actual GEE export names the city column "station", not "city" -- handle both.
    city_col = "station" if "station" in df.columns else "city"
    if city_col not in df.columns:
        raise ValueError(f"{fname}: no 'station'/'city' column found — check the GEE export script's output format.")
    df = df.rename(columns={city_col: "city"})
    df = df.drop(columns=[c for c in ["system:index", ".geo"] if c in df.columns])
    merra2_frames.append(df)

merra2_raw = pd.concat(merra2_frames, ignore_index=True)
# GEE's ee.Date.format() defaults to UTC when no timeZone argument is given (as in
# gee_export_merra2.js) -- these naive timestamps represent UTC instants.
merra2_raw["datetime"] = pd.to_datetime(merra2_raw["datetime"])  # naive, UTC

found_cities = set(merra2_raw["city"].unique())
missing = EXPECTED_CITIES - found_cities
extra = found_cities - EXPECTED_CITIES
assert not missing, f"Missing expected NCR cities in MERRA-2 upload: {missing}"
assert not extra, f"Unexpected cities in MERRA-2 upload (old station list?): {extra}"
print(f"MERRA-2: {len(merra2_raw):,} rows across {len(found_cities)} cities — all 10 NCR cities present, no extras.")

In [ ]:
# --- PM2.5 derivation, WITH a sanity check before you proceed ---
# Set this once you know which band you actually pulled (SO4CMASS vs SO4SMASS).
PM25_FORMULA = "thesis"   # "thesis" (SO4CMASS, as originally written) or "nasa_gmao" (SO4SMASS)

if PM25_FORMULA == "thesis":
    assert "SO4CMASS" in merra2_raw.columns, "PM25_FORMULA='thesis' needs a SO4CMASS column."
    merra2_raw["pm25"] = derive_pm25_thesis_formula(merra2_raw)
elif PM25_FORMULA == "nasa_gmao":
    assert "SO4SMASS" in merra2_raw.columns, "PM25_FORMULA='nasa_gmao' needs a SO4SMASS column — re-pull MERRA-2 with that band first."
    merra2_raw["pm25"] = derive_pm25_nasa_gmao_formula(merra2_raw)
else:
    raise ValueError("PM25_FORMULA must be 'thesis' or 'nasa_gmao'")

mean_pm25 = merra2_raw["pm25"].mean()
print(f"Derived PM2.5 — mean: {mean_pm25:.2f} µg/m³, min: {merra2_raw['pm25'].min():.2f}, max: {merra2_raw['pm25'].max():.2f}")

if mean_pm25 > 250:
    print("\n" + "="*70)
    print("⚠️  SANITY CHECK FAILED: mean PM2.5 is above the 'Hazardous' AQI ")
    print("    threshold for your ENTIRE dataset. This is almost certainly the")
    print("    SO4CMASS-vs-SO4SMASS issue we found earlier — do not proceed to")
    print("    training on this. Re-pull MERRA-2 with SO4SMASS and switch")
    print("    PM25_FORMULA to 'nasa_gmao' above.")
    print("="*70)

In [ ]:
# --- ERA5: upload the re-pulled file and verify it's the correct (new) city list ---
era5_uploaded = files.upload()
era5_fname = list(era5_uploaded.keys())[0]
era5_raw = pd.read_csv(era5_fname, parse_dates=["datetime"])

# normalize whatever the station column is named, and normalize "Quezon City" -> "Quezon_City" etc.
station_col = "station" if "station" in era5_raw.columns else "city"
era5_raw = era5_raw.rename(columns={station_col: "city"})
era5_raw["city"] = era5_raw["city"].str.replace(" ", "_")

# IMPORTANT: era5_raw["datetime"] was requested with timezone="Asia/Manila" in
# data_acquisition.py, so it parses as tz-AWARE (UTC+8). merra2_raw["datetime"] is
# naive UTC (GEE's default). These do NOT represent the same clock convention --
# merging them as-is either raises a tz mismatch error or (if you strip tz blindly)
# silently pairs each PM2.5 reading with the WRONG hour of meteorology, 8 hours off.
# Fix: convert ERA5 to naive UTC to match MERRA-2's convention.
era5_raw["datetime"] = era5_raw["datetime"].dt.tz_convert("UTC").dt.tz_localize(None)

found_era5_cities = set(era5_raw["city"].unique())
missing = EXPECTED_CITIES - found_era5_cities
extra = found_era5_cities - EXPECTED_CITIES
assert not missing and not extra, (
    f"ERA5 file does not match the current NCR city list. "
    f"Missing: {missing or 'none'} | Unexpected (old station list?): {extra or 'none'}"
)
print(f"ERA5: {len(era5_raw):,} rows across {len(found_era5_cities)} cities — matches current NCR scope.")
print(f"ERA5 datetime now naive UTC, range: {era5_raw['datetime'].min()} -> {era5_raw['datetime'].max()}")

## 2b. Fill the ERA5 `boundary_layer_height` gap

Your original ERA5 pull is missing `boundary_layer_height` for Jan-Jun 2024 (a
known Open-Meteo data-availability gap, not an extraction error). This fills it
using ERA5's own `blh` variable pulled directly from the Copernicus Climate
Data Store (CDS) — the authoritative source Open-Meteo itself wraps.

**Already validated on real data (see chat):** CDS's ERA5 agrees with your
existing Open-Meteo ERA5 to within a mean bias of ~26m and ~131m noise (std) —
far tighter than the MERRA-2 PBLH alternative we tried first (79m bias that
swung from -106m to +326m by month, 300-590m noise). Documented limitation:
the CDS pull uses nearest-grid-point extraction, so your 10 cities collapse
into 2 distinct value-clusters for the filled period specifically (unlike the
smoother per-city variation Open-Meteo's own interpolation gives the rest of
your ERA5 record) — a real, smaller version of MERRA-2's identical-cities
issue, worth stating explicitly if asked.

In [ ]:
cds_uploaded = files.upload()  # era5_blh_from_cds.csv
cds_fname = list(cds_uploaded.keys())[0]
cds_blh = pd.read_csv(cds_fname, parse_dates=["datetime"])
cds_blh = cds_blh.rename(columns={"blh_cds": "PBLH"})  # reuses fill_blh_gap's existing column-name contract

found_cds_cities = set(cds_blh["city"].unique())
assert found_cds_cities == EXPECTED_CITIES, f"CDS file city mismatch: {EXPECTED_CITIES.symmetric_difference(found_cds_cities)}"

# Step 1: quantify the REAL bias on THIS data (don't reuse the number from
# chat -- recompute it fresh each run so this notebook stays correct if you
# ever re-pull either source).
era5_overlap_for_bias = era5_raw.dropna(subset=["boundary_layer_height"])[["city", "datetime", "boundary_layer_height"]]
bias_stats = compare_blh_sources(era5_overlap_for_bias, cds_blh)
print("Bias check (ERA5 Open-Meteo vs ERA5 direct-CDS, overlap months only):")
print(f"  n_overlap_hours={bias_stats['n_overlap_hours']:,}  mean_diff={bias_stats['mean_diff_merra2_minus_era5']:+.1f}m  std={bias_stats['std_diff']:.1f}m")

# Step 2: fill the gap with the bias-corrected CDS values
n_missing_before = era5_raw["boundary_layer_height"].isna().sum()
era5_raw = fill_blh_gap(era5_raw, cds_blh, bias_correction=bias_stats["mean_diff_merra2_minus_era5"])

# Step 3: clip the small number of physically-impossible negative values
# (boundary layer height cannot be negative) introduced by the bias correction.
n_negative = (era5_raw["boundary_layer_height"] < 0).sum()
era5_raw["boundary_layer_height"] = era5_raw["boundary_layer_height"].clip(lower=0)

print(f"\nFilled {n_missing_before:,} previously-missing BLH rows.")
print(f"Clipped {n_negative} negative values (out of the filled rows) up to 0.")
print(f"Remaining nulls: {era5_raw['boundary_layer_height'].isna().sum()}")
print(era5_raw["blh_source"].value_counts())

In [ ]:
# --- OpenAQ: kept separate, used only for bias validation later (not merged into training data) ---
openaq_uploaded = files.upload()
openaq_fname = list(openaq_uploaded.keys())[0]
openaq_raw = pd.read_csv(openaq_fname, parse_dates=["datetime_utc", "datetime_local"])
print(f"OpenAQ: {len(openaq_raw):,} rows, locations: {openaq_raw['location_name'].unique()}")

## 3. Merge on Stations + Datetime

In [ ]:
merged = merge_datasets(
    merra2_raw[["city", "datetime", "pm25"]],
    era5_raw[["city", "datetime"] + MET_COLS],
)
print("Merged shape:", merged.shape)
print(validate_merged(merged))

## 4. Cleaning → Normalize

**Note on ordering:** the diagram shows "Cleaning → Normalize" before "Sliding window →
split," but fitting the scaler before splitting would leak test-period statistics into
training. This notebook cleans first, **splits next, then fits the scaler on the
training portion only** — the split and the scaler-fit happen together, right after
cleaning, which achieves the same pipeline shape without the leakage. See implementation
guide Section 14 for the full reasoning.

In [ ]:
# Interpolate short gaps (<=3h) per city; report any long gaps that remain (e.g. the
# 6-month boundary_layer_height gap found earlier — these will NOT be silently fixed).
cleaned = pd.concat(
    [interpolate_gaps(g) for _, g in merged.groupby("city")], ignore_index=True
)
remaining_nulls = cleaned[FEATURE_COLS].isna().sum()
print("Remaining nulls after interpolation (long gaps, left for windowing to exclude):")
print(remaining_nulls[remaining_nulls > 0])

In [ ]:
cutoff_1, cutoff_2 = chronological_cutoffs(cleaned["datetime"])
print(f"train < {cutoff_1}  |  val: {cutoff_1} to {cutoff_2}  |  test >= {cutoff_2}")

train_df, val_df, test_df = chronological_split(cleaned, cutoff_1, cutoff_2)
scaled_train, scaled_val, scaled_test, scaler = fit_transform_split(train_df, val_df, test_df)
full_scaled = pd.concat([scaled_train, scaled_val, scaled_test], ignore_index=True)
print(f"train/val/test rows: {len(scaled_train):,} / {len(scaled_val):,} / {len(scaled_test):,}")

## 5. Sliding window (L=48) → assign to 70/10/20 split

Windows are generated per city (never crossing a city boundary) and assigned to a
split by their own start/end timestamps — any window whose span crosses a split
boundary is dropped rather than assigned to either side (guide Section 15).

In [ ]:
dataset = build_dataset(full_scaled, cutoff_1, cutoff_2, L=48, H=24)
for split, (xp, xm, y, cities) in dataset.items():
    print(f"{split:6s}  X_pm25={xp.shape}  X_met={xm.shape}  Y={y.shape}  cities={cities.shape}")

In [ ]:
# Save to Drive so Stage 2 (model training) doesn't need to re-run all of this.
import joblib

OUT_DIR = f'{PROJECT_DIR}/data/final' if 'PROJECT_DIR' in dir() else '/content/final'
os.makedirs(OUT_DIR, exist_ok=True)

for split, (xp, xm, y, cities) in dataset.items():
    np.savez(f'{OUT_DIR}/{split}.npz', X_pm25=xp, X_met=xm, Y=y, cities=cities)
joblib.dump(scaler, f'{OUT_DIR}/scaler.pkl')
print(f"Saved train/val/test .npz files and scaler.pkl to {OUT_DIR}")

## 6. Separate branch: NASA MERRA-2 bias validation against OpenAQ Manila

This does **not** feed into training — it's Section 12 of the implementation guide:
characterizing how well MERRA-2-derived PM2.5 tracks real ground-sensor readings over
the Manila overlap window (2023-09-06 to 2023-10-23).

In [ ]:
from scipy import stats

manila_sensor = openaq_raw[openaq_raw["location_name"] == "Manila"].copy()
manila_sensor = manila_sensor[manila_sensor["pm25_ugm3"] <= 200]  # drop sensor artifacts (Badura et al., 2019)
# Use datetime_utc directly -- it's already UTC, matching merra2_raw's convention,
# with no timezone conversion needed (avoids the same UTC/+08:00 pitfall as Cell 8).
manila_sensor["datetime"] = pd.to_datetime(manila_sensor["datetime_utc"]).dt.tz_convert("UTC").dt.tz_localize(None)

manila_model = merra2_raw[merra2_raw["city"] == "Manila"][["datetime", "pm25"]].copy()

val = manila_sensor.merge(manila_model, on="datetime", how="inner", suffixes=("_sensor", "_model"))
val = val.rename(columns={"pm25_ugm3": "pm25_sensor", "pm25": "pm25_model"})
val = val[(val["datetime"] >= "2023-09-06") & (val["datetime"] <= "2023-10-23")]

openaq_validation = compute_bias_metrics(val["pm25_model"], val["pm25_sensor"])
print(f"n = {openaq_validation['n']} overlapping hourly readings")
print(f"Pearson r = {openaq_validation['pearson_r']:.3f}")
print(f"MBE       = {openaq_validation['mbe']:+.2f} \u00b5g/m\u00b3")
print(f"RMSE      = {openaq_validation['rmse']:.2f} \u00b5g/m\u00b3")
print("\nReminder: this is the MERRA-2-specific number \u2014 don't reuse the old CAMS-era")
print("r=0.537/0.793, MBE=+0.07 figures from earlier drafts (guide Section 12 / item 13).")

# Save so it's not lost when the Colab session ends (previously printed-only).
import json
val_dir = f'{PROJECT_DIR}/results/statistical_tests' if 'PROJECT_DIR' in dir() else '/content/results'
os.makedirs(val_dir, exist_ok=True)
with open(f'{val_dir}/openaq_manila_validation.json', 'w') as f:
    json.dump(openaq_validation, f, indent=2)
print(f"\nSaved to {val_dir}/openaq_manila_validation.json")

## 7. EMB-DENR annual PM2.5 comparison — attempted, not viable

**Checked and abandoned.** EMB-DENR's public Power BI dashboard (air.emb.gov.ph) was
filtered to Region = NCR for all ten study cities. Every single one shows the same
pattern: annual PM2.5 figures exist for roughly 2013–2017, then nothing — no bars at
all for 2018 onward, including the entire Aug 2022–Dec 2024 study period. This is
consistent across all 10 cities, not a per-city gap, indicating the dashboard's NCR
annual reporting itself stopped around 2017–2018 rather than simply lacking recent
data for a few stations.

**Conclusion:** no valid EMB-DENR annual comparison exists for this study's time
window. Comparing 2022–2024 MERRA-2 data against a 2013–2017 EMB-DENR figure would
not be a validation of anything — the two would differ for reasons having nothing to
do with data quality (different years, different pollution conditions, monsoon
patterns, emissions changes over ~7 years). No comparison table was built.

**This finding is itself citable evidence, not a dead end.** It reinforces the same
point made elsewhere in Ethical Considerations regarding EMB-DENR's ground-monitoring
limitations: `air.emb.gov.ph`'s own public annual dashboard has no NCR PM2.5 records
covering this study's 2022–2024 period, for any of the ten cities studied — a
documented, screenshot-able gap in official public reporting, independent of and
additional to the earlier finding that EMB-DENR could only provide monthly/
quarterly/annual (not hourly) data even where it exists (IIS-2026-017941).

OpenAQ Manila (Section 6 above) remains the study's only ground-truth bias check.

## Next: Stage 2 — Model Training

Load the four `.npz` files from `data/final/` and follow `src/training/train.py` for
each of the four variants (A, B, C/AIRPRED, D) — same code, same config, run
sequentially. Happy to build that as a second Colab notebook once Stage 1 output here
looks right (i.e., once the PM2.5 sanity check in Cell 5 passes).